# ChatGPT Conversation History - Temporal Analysis

This notebook analyzes how your ChatGPT usage has evolved over time.

**Focus Areas:**
- Usage patterns over time
- Trends and seasonality
- Activity by day of week and time
- Evolution of conversation complexity

---

## Setup

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from collections import Counter
import calendar
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries loaded!")

In [ ]:
# Load data
import os
if not os.path.exists('conversations.json'):
    raise FileNotFoundError("conversations.json not found — put it in the repo root or update the path.")
with open('conversations.json', 'r', encoding='utf-8') as f:
    conversations = json.load(f)

# Load the DataFrame we created in notebook 01
df = pd.read_csv('../chatgpt_conversations_dataframe.csv')
df['create_date'] = pd.to_datetime(df['create_date'])

print(f"Loaded {len(conversations):,} conversations")
print(f"DataFrame shape: {df.shape}")

## 1. Conversations Over Time

In [ ]:
# Daily conversation counts
daily_counts = df.groupby(df['create_date'].dt.date).size()

# Weekly conversation counts
df['week'] = df['create_date'].dt.to_period('W')
weekly_counts = df.groupby('week').size()

# Monthly conversation counts
df['month'] = df['create_date'].dt.to_period('M')
monthly_counts = df.groupby('month').size()

print("TEMPORAL AGGREGATIONS")
print(f"Daily average: {daily_counts.mean():.2f} conversations")
print(f"Weekly average: {weekly_counts.mean():.2f} conversations")
print(f"Monthly average: {monthly_counts.mean():.2f} conversations")

In [ ]:
# Visualize conversations over time
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Daily
axes[0].plot(daily_counts.index, daily_counts.values, alpha=0.6, linewidth=1)
axes[0].plot(daily_counts.index, daily_counts.rolling(window=7, center=True).mean(), 
             color='red', linewidth=2, label='7-day moving average')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Conversations')
axes[0].set_title('Daily Conversation Count', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Weekly
weekly_dates = [p.start_time for p in weekly_counts.index]
axes[1].bar(weekly_dates, weekly_counts.values, width=6, edgecolor='black', alpha=0.7)
axes[1].axhline(weekly_counts.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {weekly_counts.mean():.1f}')
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Conversations')
axes[1].set_title('Weekly Conversation Count', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Monthly
monthly_dates = [p.start_time for p in monthly_counts.index]
bars = axes[2].bar(monthly_dates, monthly_counts.values, width=25, edgecolor='black')
axes[2].axhline(monthly_counts.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {monthly_counts.mean():.1f}')

# Color bars by value
colors = plt.cm.viridis(monthly_counts.values / monthly_counts.max())
for bar, color in zip(bars, colors):
    bar.set_color(color)

axes[2].set_xlabel('Month')
axes[2].set_ylabel('Conversations')
axes[2].set_title('Monthly Conversation Count', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Find most and least active periods
print("ACTIVITY EXTREMES")
print("\nMost Active:")
print(f"  Day: {daily_counts.idxmax()} ({daily_counts.max()} conversations)")
print(f"  Week: {weekly_counts.idxmax()} ({weekly_counts.max()} conversations)")
print(f"  Month: {monthly_counts.idxmax()} ({monthly_counts.max()} conversations)")

print("\nLeast Active:")
print(f"  Day: {daily_counts.idxmin()} ({daily_counts.min()} conversations)")
print(f"  Week: {weekly_counts.idxmin()} ({weekly_counts.min()} conversations)")
print(f"  Month: {monthly_counts.idxmin()} ({monthly_counts.min()} conversations)")

## 2. Day of Week Analysis

In [ ]:
# Add day of week
df['day_of_week'] = df['create_date'].dt.day_name()
df['day_of_week_num'] = df['create_date'].dt.dayofweek

# Count by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = df['day_of_week'].value_counts().reindex(day_order)

print("CONVERSATIONS BY DAY OF WEEK")
print("=" * 50)
for day, count in dow_counts.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct)
    print(f"{day:<12} {count:>4} ({pct:>5.1f}%) {bar}")
print("=" * 50)

In [ ]:
# Visualize day of week patterns
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
colors_dow = ['#ff6b6b' if day in ['Saturday', 'Sunday'] else '#4ecdc4' for day in day_order]
bars = axes[0].bar(day_order, dow_counts.values, edgecolor='black', color=colors_dow)
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Conversation Count')
axes[0].set_title('Conversations by Day of Week', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, count in zip(bars, dow_counts.values):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}', ha='center', va='bottom', fontweight='bold')

# Pie chart
weekday_count = dow_counts[['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']].sum()
weekend_count = dow_counts[['Saturday', 'Sunday']].sum()

axes[1].pie([weekday_count, weekend_count], 
           labels=['Weekday', 'Weekend'],
           autopct='%1.1f%%',
           colors=['#4ecdc4', '#ff6b6b'],
           startangle=90)
axes[1].set_title('Weekday vs Weekend', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nWeekday vs Weekend:")
print(f"  Weekday: {weekday_count} conversations ({weekday_count/len(df)*100:.1f}%)")
print(f"  Weekend: {weekend_count} conversations ({weekend_count/len(df)*100:.1f}%)")

## 3. Hour of Day Analysis

In [ ]:
# Extract hour from create_date
df['hour'] = df['create_date'].dt.hour

# Count by hour
hour_counts = df['hour'].value_counts().sort_index()

print("CONVERSATIONS BY HOUR OF DAY")
print("=" * 60)
for hour in range(24):
    count = hour_counts.get(hour, 0)
    pct = (count / len(df)) * 100 if count > 0 else 0
    bar = '█' * int(pct * 2)
    period = 'AM' if hour < 12 else 'PM'
    display_hour = hour if hour <= 12 else hour - 12
    if display_hour == 0:
        display_hour = 12
    print(f"{display_hour:>2}:00 {period}  {count:>4} ({pct:>4.1f}%) {bar}")
print("=" * 60)

In [ ]:
# Visualize hourly pattern
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Line plot
axes[0].plot(hour_counts.index, hour_counts.values, marker='o', linewidth=2, markersize=8)
axes[0].fill_between(hour_counts.index, hour_counts.values, alpha=0.3)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Conversation Count')
axes[0].set_title('Conversations by Hour of Day', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(24))
axes[0].grid(True, alpha=0.3)

# Add time period labels
axes[0].axvspan(0, 6, alpha=0.1, color='blue', label='Night (12am-6am)')
axes[0].axvspan(6, 12, alpha=0.1, color='yellow', label='Morning (6am-12pm)')
axes[0].axvspan(12, 18, alpha=0.1, color='orange', label='Afternoon (12pm-6pm)')
axes[0].axvspan(18, 24, alpha=0.1, color='purple', label='Evening (6pm-12am)')
axes[0].legend(loc='upper left')

# Polar plot (clock visualization)
ax_polar = plt.subplot(2, 1, 2, projection='polar')
theta = np.linspace(0, 2 * np.pi, 24, endpoint=False)
radii = hour_counts.values
width = 2 * np.pi / 24

bars = ax_polar.bar(theta, radii, width=width, bottom=0, edgecolor='black')

# Color by value
colors = plt.cm.plasma(radii / radii.max())
for bar, color in zip(bars, colors):
    bar.set_color(color)

ax_polar.set_theta_zero_location('N')
ax_polar.set_theta_direction(-1)
ax_polar.set_xticks(theta)
ax_polar.set_xticklabels([f'{h:02d}:00' for h in range(24)])
ax_polar.set_title('24-Hour Activity Pattern (Clock View)', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Time period breakdown
def get_time_period(hour):
    if 0 <= hour < 6:
        return 'Night (12am-6am)'
    elif 6 <= hour < 12:
        return 'Morning (6am-12pm)'
    elif 12 <= hour < 18:
        return 'Afternoon (12pm-6pm)'
    else:
        return 'Evening (6pm-12am)'

df['time_period'] = df['hour'].apply(get_time_period)
period_counts = df['time_period'].value_counts()

print("\nCONVERSATIONS BY TIME PERIOD")
print("=" * 50)
for period in ['Morning (6am-12pm)', 'Afternoon (12pm-6pm)', 'Evening (6pm-12am)', 'Night (12am-6am)']:
    count = period_counts.get(period, 0)
    pct = (count / len(df)) * 100
    print(f"{period:<25} {count:>4} ({pct:>5.1f}%)")
print("=" * 50)

## 4. Heatmap: Day of Week vs Hour

In [ ]:
# Create heatmap data
heatmap_data = df.groupby(['day_of_week_num', 'hour']).size().unstack(fill_value=0)

# Reindex to ensure all hours are present
heatmap_data = heatmap_data.reindex(columns=range(24), fill_value=0)
heatmap_data.index = day_order

# Create heatmap
plt.figure(figsize=(16, 6))
sns.heatmap(heatmap_data, annot=True, fmt='g', cmap='YlOrRd', 
            cbar_kws={'label': 'Conversation Count'}, linewidths=0.5)
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.title('Conversation Activity Heatmap: Day of Week vs Hour', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find peak activity time
max_val = heatmap_data.max().max()
peak_day, peak_hour = [(day, hour) for day in heatmap_data.index 
                       for hour in heatmap_data.columns 
                       if heatmap_data.loc[day, hour] == max_val][0]

print(f"\nPeak Activity: {peak_day}s at {peak_hour}:00 ({int(max_val)} conversations)")

## 5. Evolution of Conversation Complexity

In [ ]:
# Analyze how conversation length has changed over time
monthly_stats = df.groupby('month').agg({
    'message_count': ['mean', 'median', 'max'],
    'user_messages': 'sum',
    'assistant_messages': 'sum',
    'tool_messages': 'sum'
}).reset_index()

monthly_stats.columns = ['month', 'avg_length', 'median_length', 'max_length', 
                        'total_user', 'total_assistant', 'total_tool']
monthly_stats['month_date'] = monthly_stats['month'].apply(lambda x: x.start_time)

print("CONVERSATION COMPLEXITY OVER TIME")
display(monthly_stats[['month', 'avg_length', 'median_length', 'max_length']].tail(12))

In [ ]:
# Visualize evolution
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Average and median length over time
axes[0].plot(monthly_stats['month_date'], monthly_stats['avg_length'], 
            marker='o', linewidth=2, label='Average Length', color='blue')
axes[0].plot(monthly_stats['month_date'], monthly_stats['median_length'], 
            marker='s', linewidth=2, label='Median Length', color='green')
axes[0].fill_between(monthly_stats['month_date'], monthly_stats['avg_length'], 
                     monthly_stats['median_length'], alpha=0.2)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Messages per Conversation')
axes[0].set_title('Evolution of Conversation Length', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Message type distribution over time
axes[1].plot(monthly_stats['month_date'], monthly_stats['total_user'], 
            marker='o', linewidth=2, label='User Messages')
axes[1].plot(monthly_stats['month_date'], monthly_stats['total_assistant'], 
            marker='s', linewidth=2, label='Assistant Messages')
axes[1].plot(monthly_stats['month_date'], monthly_stats['total_tool'], 
            marker='^', linewidth=2, label='Tool Messages')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Total Messages')
axes[1].set_title('Message Volume by Type Over Time', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Growth and Trends

In [ ]:
# Cumulative conversations over time
df_sorted = df.sort_values('create_date')
df_sorted['cumulative_count'] = range(1, len(df_sorted) + 1)

# Calculate growth rate
monthly_growth = monthly_counts.pct_change() * 100

print("GROWTH ANALYSIS")
print(f"\nTotal Growth: From 1 to {len(df):,} conversations")
print(f"\nRecent Monthly Growth Rates:")
for month, rate in monthly_growth.tail(6).items():
    if not np.isnan(rate):
        direction = "📈" if rate > 0 else "📉"
        print(f"  {month}: {direction} {rate:+.1f}%")

In [ ]:
# Visualize cumulative growth
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Cumulative conversations
axes[0].plot(df_sorted['create_date'], df_sorted['cumulative_count'], linewidth=2)
axes[0].fill_between(df_sorted['create_date'], df_sorted['cumulative_count'], alpha=0.3)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Cumulative Conversations')
axes[0].set_title('Cumulative Conversation Growth', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Monthly growth rate
growth_data = monthly_growth.dropna()
colors = ['green' if x > 0 else 'red' for x in growth_data.values]
axes[1].bar([p.start_time for p in growth_data.index], growth_data.values, 
           color=colors, edgecolor='black', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Growth Rate (%)')
axes[1].set_title('Month-over-Month Growth Rate', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Temporal Insights Summary

In [ ]:
# Find most productive day and time
most_active_dow = dow_counts.idxmax()
most_active_hour = hour_counts.idxmax()
most_active_period = period_counts.idxmax()

print("\n" + "=" * 70)
print("TEMPORAL INSIGHTS SUMMARY")
print("=" * 70)

print(f"\n📅 PEAK ACTIVITY TIMES")
print(f"   • Most active day: {most_active_dow} ({dow_counts[most_active_dow]} conversations)")
print(f"   • Most active hour: {most_active_hour}:00 ({hour_counts[most_active_hour]} conversations)")
print(f"   • Most active period: {most_active_period} ({period_counts[most_active_period]} conversations)")
print(f"   • Peak moment: {peak_day}s at {peak_hour}:00")

print(f"\n📊 USAGE PATTERNS")
weekday_pct = (weekday_count / len(df)) * 100
print(f"   • Weekday usage: {weekday_pct:.1f}%")
print(f"   • Weekend usage: {100-weekday_pct:.1f}%")
if weekday_pct > 75:
    print(f"   • Strong weekday preference - work/productivity focus")
elif weekday_pct < 60:
    print(f"   • Balanced weekday/weekend usage")

print(f"\n📈 TRENDS")
recent_avg = monthly_counts.tail(3).mean()
older_avg = monthly_counts.head(3).mean()
trend_pct = ((recent_avg - older_avg) / older_avg * 100) if older_avg > 0 else 0
print(f"   • Early period avg: {older_avg:.1f} conversations/month")
print(f"   • Recent period avg: {recent_avg:.1f} conversations/month")
if trend_pct > 10:
    print(f"   • 📈 Usage is increasing ({trend_pct:+.1f}% growth)")
elif trend_pct < -10:
    print(f"   • 📉 Usage is decreasing ({trend_pct:+.1f}% change)")
else:
    print(f"   • ➡️ Usage is relatively stable")

print(f"\n🎯 CONVERSATION EVOLUTION")
recent_complexity = monthly_stats['avg_length'].tail(3).mean()
older_complexity = monthly_stats['avg_length'].head(3).mean()
complexity_change = recent_complexity - older_complexity
print(f"   • Early conversations: {older_complexity:.1f} messages avg")
print(f"   • Recent conversations: {recent_complexity:.1f} messages avg")
if complexity_change > 2:
    print(f"   • Conversations are becoming more in-depth (+{complexity_change:.1f} messages)")
elif complexity_change < -2:
    print(f"   • Conversations are becoming more concise ({complexity_change:.1f} messages)")
else:
    print(f"   • Conversation complexity is stable")

print("\n" + "=" * 70)
print("Next: Run notebook 03 for topic analysis and clustering")
print("=" * 70)